# Notebook 04 — Modélisation
## Phase 3 : Modélisation, rééquilibrage et comparaison des modèles

Objectif : entraîner au moins 4 familles de modèles, tester au moins 3 stratégies de gestion du déséquilibre, et identifier la meilleure combinaison modèle × stratégie selon la métrique F1.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
import joblib

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
X_train = train_df.drop(columns=['bad_nutrition'])
y_train = train_df['bad_nutrition']

preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')

print('Train shape:', X_train.shape)
print('Target distribution:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight pour XGBoost : {scale_pos_weight:.2f}')


Train shape: (9068, 16)
Target distribution:
bad_nutrition
0    82.49
1    17.51
Name: proportion, dtype: float64
scale_pos_weight pour XGBoost : 4.71


## 1. Modèles sélectionnés

Nous testons quatre familles de modèles diversifiées :
- Régression logistique (baseline linéaire, class_weight)
- Arbre de décision (non-linéaire, interprétable)
- Forêt aléatoire (ensemble, robuste)
- XGBoost (ensemble boosté, performant sur données tabulaires)

Ces quatre types couvrent des approches linéaires, arborescentes, ensemblistes et boostées.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(random_state=42, solver='liblinear', max_iter=5000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(random_state=42, n_estimators=200, class_weight='balanced_subsample'),
    'XGBoost': XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, eval_metric='logloss')
}

strategies = {
    'baseline': None,
    'smote': SMOTE(random_state=42),
    'undersample': RandomUnderSampler(random_state=42),
    'smote_tomek': SMOTETomek(random_state=42)   
}

def build_pipeline(model, sampler=None):
    preproc = clone(preprocessor)
    if sampler is None:
        return Pipeline([('preprocessor', preproc), ('clf', clone(model))])
    return ImbPipeline([('preprocessor', preproc), ('sampler', sampler), ('clf', clone(model))])

def evaluate_pipeline(pipeline):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=1)
    return scores.mean(), scores.std()

In [5]:
results = []
for model_name, model in models.items():
    for strategy_name, sampler in strategies.items():
        pipe = build_pipeline(model, sampler)
        mean_f1, std_f1 = evaluate_pipeline(pipe)
        results.append({
            'model': model_name,
            'strategy': strategy_name,
            'mean_f1': mean_f1,
            'std_f1': std_f1
        })
        print(f'{model_name:<20} | {strategy_name:<12} | F1 = {mean_f1:.4f} ± {std_f1:.4f}')
results_df = pd.DataFrame(results).sort_values(by=['mean_f1', 'std_f1'], ascending=[False, True]).reset_index(drop=True)
results_df.to_csv(MODEL_DIR / 'model_selection_results.csv', index=False)

print('Meilleure configuration :')
print(results_df.head(1).to_string(index=False))

LogisticRegression   | baseline     | F1 = 0.6952 ± 0.0191
LogisticRegression   | smote        | F1 = 0.7008 ± 0.0209
LogisticRegression   | undersample  | F1 = 0.6806 ± 0.0156
LogisticRegression   | smote_tomek  | F1 = 0.7003 ± 0.0216
DecisionTree         | baseline     | F1 = 0.8139 ± 0.0329
DecisionTree         | smote        | F1 = 0.8143 ± 0.0206
DecisionTree         | undersample  | F1 = 0.7757 ± 0.0247
DecisionTree         | smote_tomek  | F1 = 0.8155 ± 0.0197
RandomForest         | baseline     | F1 = 0.8542 ± 0.0256
RandomForest         | smote        | F1 = 0.8638 ± 0.0203
RandomForest         | undersample  | F1 = 0.8186 ± 0.0198
RandomForest         | smote_tomek  | F1 = 0.8625 ± 0.0206


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:06] WARNING: C:\actions-r

XGBoost              | baseline     | F1 = 0.8785 ± 0.0243


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:08] WARNING: C:\actions-r

XGBoost              | smote        | F1 = 0.8612 ± 0.0192


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:09] WARNING: C:\actions-r

XGBoost              | undersample  | F1 = 0.8132 ± 0.0162


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:20] WARNING: C:\actions-r

XGBoost              | smote_tomek  | F1 = 0.8624 ± 0.0243
Meilleure configuration :
  model strategy  mean_f1   std_f1
XGBoost baseline  0.87854 0.024338


c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:08:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [6]:
# Tableau comparatif 4 modèles × 4 stratégies
table = pd.DataFrame(index=results_df['model'].unique())

for strategy in ['baseline', 'smote', 'undersample', 'smote_tomek']:
    col_values = []
    for model in table.index:
        row = results_df[(results_df['model'] == model) & (results_df['strategy'] == strategy)]
        if not row.empty:
            f1 = row['mean_f1'].values[0]
            std = row['std_f1'].values[0]
            col_values.append(f'{f1:.4f} ± {std:.4f}')
        else:
            col_values.append('N/A')
    table[strategy] = col_values

print(table.to_string())

                           baseline            smote      undersample      smote_tomek
XGBoost             0.8785 ± 0.0243  0.8612 ± 0.0192  0.8132 ± 0.0162  0.8624 ± 0.0243
RandomForest        0.8542 ± 0.0256  0.8638 ± 0.0203  0.8186 ± 0.0198  0.8625 ± 0.0206
DecisionTree        0.8139 ± 0.0329  0.8143 ± 0.0206  0.7757 ± 0.0247  0.8155 ± 0.0197
LogisticRegression  0.6952 ± 0.0191  0.7008 ± 0.0209  0.6806 ± 0.0156  0.7003 ± 0.0216


## 2. Analyse et choix du meilleur modèle

Les résultats F1±σ pour les 16 configurations (4 modèles × 4 stratégies) sont sauvegardés dans `models/model_selection_results.csv`.
Le modèle retenu pour l'optimisation en Phase 3 sera celui qui maximise la moyenne F1.
La meilleure combinaison identifiée est **XGBoost + baseline** avec F1 = 0.8785 ± 0.0243.